### **MODELING**

In this section, we will focus on building and evaluating predictive models for Remaining Useful Life (RUL):

1. **Baseline Model:**

   * We will first implement a **Linear Regression model** as a baseline.
   * This will help us evaluate the impact of scaling and feature engineering on model performance.

2. **Advanced Models:**

   * After establishing the baseline, we will model the data using more advanced algorithms:

     * **XGBoost**
     * **NGBoost**
     * **Decision Trees**
   * These models are chosen for their ability to capture non-linear relationships and complex feature interactions.

3. **Model Evaluation:**

   * The performance of each model will be assessed using regression metrics such as:

     * **Root Mean Squared Error (RMSE)**
     * **Mean Absolute Error (MAE)**
     * **R² Score**
   * This comparison will allow us to identify the best-performing model.

4. **Model Optimization:**

   * Once the best model is selected, we will perform **hyperparameter tuning** to further improve its predictive accuracy and generalization ability.

**The modeling phase will provide both a baseline and advanced approaches, enabling us to compare performance across different methods and identify the optimal model to predict RUL effectively.**




In [5]:
 # Importing necessary libraries
import pandas as pd
from xgboost import XGBRegressor
from sklearn.model_selection import  GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import Ridge
from ngboost import NGBRegressor
from ngboost.distns import LogNormal, Poisson
from sklearn.metrics import accuracy_score, confusion_matrix , precision_score , recall_score , f1_score , r2_score 
from sklearn.metrics import mean_squared_error 
from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.model_selection import cross_val_score,KFold


In [2]:
 # Loading the datasets 
cleaned_test_dfs = {}
cleaned_train_dfs = {}
train_dfs = {}
test_dfs = {}

names = ['FD001','FD002','FD003','FD004']
test_cleaned_csv = ['cleaned_test FD001 with RUL.csv','cleaned_test FD002 with RUL.csv','cleaned_test FD003 with RUL.csv','cleaned_test FD004 with RUL.csv']
train_cleaned_csv = ['cleaned_train FD001 with RUL.csv','cleaned_train FD002 with RUL.csv','cleaned_train FD003 with RUL.csv','cleaned_train FD004 with RUL.csv']

test_csv = ['test FD001 with RUL.csv','test FD002 with RUL.csv','test FD003 with RUL.csv','test FD004 with RUL.csv']
train_csv = ['train FD001 with RUL.csv','train FD002 with RUL.csv','train FD003 with RUL.csv','train FD004 with RUL.csv']

 # loading the datasets
for name , df1 , df2 , df3 , df4 in zip( names, test_cleaned_csv, train_cleaned_csv, test_csv, train_csv ):
    cleaned_test_dfs[name] = pd.read_csv(df1)
    cleaned_train_dfs[name] = pd.read_csv(df2)
    test_dfs[name] = pd.read_csv(df3)
    train_dfs[name] = pd.read_csv(df4)


we will first model the unscalled and unfeatured data to see how the model performs with unscalled data then compare the model performance with the scalled data.

In [3]:
 # pipeline
# Cross-validation setup
cv = KFold(n_splits=5, shuffle=True, random_state=42)

for (name1, df1) , (name2, df2)in zip(train_dfs.items(), test_dfs.items()):
    X_train = df1.iloc[ :,1:-1 ]
    y_train = df1.iloc[ :,-1 ]

    X_test = df2.iloc[ :,1:-1 ]
    y_test = df2.iloc[ :,-1 ]

     # --fitting a baseline model-- 
    reg1 = LinearRegression()
    reg1.fit(X_train, y_train )
    y_pred = reg1.predict( X_test )

    print(f'========= {name1} MODEL OUTPUTS ==========')
    print('R2 Score(Unscaled) :', r2_score( y_test, y_pred ))
    mse_baseline_unscalled = mean_squared_error(y_test, y_pred)
    print(f"MSE(Unscaled): {mse_baseline_unscalled}")

     # -- Cross Validation --
    cv_scores = cross_val_score(reg1, X_train, y_train, cv=cv, scoring='r2')
    print('Linear Regression (Unscaled) Cross validation R2 Mean :', cv_scores.mean())

     # --Ridge regression for regulization--
    ridge = Ridge(alpha= 1.0 )
    ridge.fit(X_train, y_train)
    y_pred_ridge = ridge.predict(X_test)
    
    print(f'Ridge Regression R2 Score(Unscaled) :', r2_score(y_test, y_pred_ridge))
    mse_ridge_unscalled = mean_squared_error(y_test, y_pred_ridge)
    print(f"MSE_ridge(Unscaled): {mse_ridge_unscalled}")
    
     # --Scalling the data--
    scaler = StandardScaler().set_output( transform = 'pandas')
    scaler.fit(X_train)

    scaled_X_train = scaler.transform(X_train)
    scaled_X_test = scaler.transform(X_test)
    
    reg2 = LinearRegression()
    reg2.fit(scaled_X_train,y_train)
    y_pred_scaled = reg2.predict(scaled_X_test)

    print('R2 Score(Scaled):', r2_score( y_test, y_pred_scaled ))
    mse_baseline_scaled = mean_squared_error(y_test, y_pred_scaled)
    print(f"MSE(Scaled): {mse_baseline_scaled}")
    
    # --Cross validation--
    cv_scores = cross_val_score(reg2, scaled_X_train, y_train, cv=cv, scoring='r2')
    print('Linear Regression (Scaled) Cross validation R2 Mean :', cv_scores.mean())
    
     # --Ridge regression for regulization--
    ridge = Ridge(alpha= 1.0 )
    ridge.fit(scaled_X_train, y_train)
    y_pred_ridge_scaled = ridge.predict(scaled_X_test)
    
    print(f'Ridge Regression (Scaled) R2 Score :', r2_score(y_test, y_pred_ridge_scaled))
    mse_ridge_scaled = mean_squared_error(y_test, y_pred_ridge_scaled)
    print(f"MSE_Rige(Scaled): {mse_ridge_scaled}")
    print('---'*50)


========= FD001 MODEL OUTPUTS ==========
R2 Score(Unscaled) : 0.6972198889255656
MSE(Unscaled): 522.8617692990639
Linear Regression (Unscaled) Cross validation R2 Mean : 0.7653736260953764
Ridge Regression R2 Score(Unscaled) : 0.6983297483170029
MSE_ridge(Unscaled): 520.9451868557164
R2 Score(Scaled): 0.6972198889255832
MSE(Scaled): 522.8617692990337
Linear Regression (Scaled) Cross validation R2 Mean : 0.76537362609538
Ridge Regression (Scaled) R2 Score : 0.6972239327641268
MSE_Rige(Scaled): 522.8547861171855
------------------------------------------------------------------------------------------------------------------------------------------------------
========= FD002 MODEL OUTPUTS ==========
R2 Score(Unscaled) : 0.6766257384518963
MSE(Unscaled): 935.2496850749297
Linear Regression (Unscaled) Cross validation R2 Mean : 0.7510060200153571
Ridge Regression R2 Score(Unscaled) : 0.6773250012838155
MSE_ridge(Unscaled): 933.2273059894515
R2 Score(Scaled): 0.6766257384518336
MSE(Scaled)

From the baseline experiments, **FD001** dataset is  the easiest dataset to model, achieving the **highest R² and lowest prediction error**, while **FD004** proves to be the most challenging due to its complex operating conditions and higher noise levels. **Ridge regression does not** consistently improve performance , while it provides a slight benefit in **FD003**, it  **reduces  the accuracy in FD004**, suggesting that regularization may sometimes remove informative variance. **Scaling  has no impact on model performance**. 

We will then model the datasets where we removed the low variance columns to see the effect on the models performance.


In [4]:
 # pipeline
# Cross-validation setup
cv = KFold(n_splits=5, shuffle=True, random_state=42)
for (name1, df1) , (name2, df2)in zip(cleaned_train_dfs.items(), cleaned_test_dfs.items()):
    X_train = df1.iloc[ :,1:-1 ]
    y_train = df1.iloc[ :,-1 ]

    X_test = df2.iloc[ :,1:-1 ]
    y_test = df2.iloc[ :,-1 ]

     # --fitting a baseline model-- 
    reg3 = LinearRegression()
    reg3.fit(X_train, y_train )
    y_pred = reg3.predict( X_test )

    print(f'========= {name1} MODEL OUTPUTS ==========')
    print('R2 Score(Unscaled) :', r2_score( y_test, y_pred ))
    mse_baseline_unscalled = mean_squared_error(y_test, y_pred)
    print(f"MSE(Scaled): {mse_baseline_unscalled}")

     # -- Cross Validation --
    cv_scores = cross_val_score(reg3, X_train, y_train, cv=cv, scoring='r2')
    print('Linear Regression (Unscaled) Cross validation R2 Mean :', cv_scores.mean())


     # --Ridge regression for regulization--
    ridge = Ridge(alpha= 1.0 )
    ridge.fit(X_train, y_train)
    y_pred_ridge = ridge.predict(X_test)
    
    print(f'Ridge Regression R2 Score(Unscaled) :', r2_score(y_test, y_pred_ridge))
    mse_ridge_unscalled = mean_squared_error(y_test, y_pred_ridge)
    print(f"MSE_ridge(Unscaled): {mse_ridge_unscalled}")
    
     # --Scalling the data--
    scaler = StandardScaler().set_output( transform = 'pandas')
    scaler.fit(X_train)

    scaled_X_train = scaler.transform(X_train)
    scaled_X_test = scaler.transform(X_test)
    
    reg4 = LinearRegression()
    reg4.fit(scaled_X_train,y_train)
    y_pred_scaled = reg4.predict(scaled_X_test)

    print('R2 Score(Scaled):', r2_score( y_test, y_pred_scaled ))
    mse_baseline_scaled = mean_squared_error(y_test, y_pred_scaled)
    print(f"MSE(Scaled): {mse_baseline_scaled}")
    
    # --Cross validation--
    cv_scores = cross_val_score(reg4, scaled_X_train, y_train, cv=cv, scoring='r2')
    print('Linear Regression (Scaled) Cross validation R2 Mean :', cv_scores.mean())
    
     # --Ridge regression for regulization--
    ridge = Ridge(alpha= 1.0 )
    ridge.fit(scaled_X_train, y_train)
    y_pred_ridge_scaled = ridge.predict(scaled_X_test)
    
    print(f'Ridge Regression (Scaled) R2 Score :', r2_score(y_test, y_pred_ridge_scaled))
    mse_ridge_scaled = mean_squared_error(y_test, y_pred_ridge_scaled)
    print(f"MSE_Rige(Scaled): {mse_ridge_scaled}")
    print('---'*50)


========= FD001 MODEL OUTPUTS ==========
R2 Score(Unscaled) : 0.6970361032923211
MSE(Scaled): 523.1791431220305
Linear Regression (Unscaled) Cross validation R2 Mean : 0.7631424734552641
Ridge Regression R2 Score(Unscaled) : 0.6969431081554582
MSE_ridge(Unscaled): 523.339733596827
R2 Score(Scaled): 0.6970361032923144
MSE(Scaled): 523.1791431220423
Linear Regression (Scaled) Cross validation R2 Mean : 0.763142473455264
Ridge Regression (Scaled) R2 Score : 0.6970400859415126
MSE_Rige(Scaled): 523.1722656062145
------------------------------------------------------------------------------------------------------------------------------------------------------
========= FD002 MODEL OUTPUTS ==========
R2 Score(Unscaled) : 0.6777829618269525
MSE(Scaled): 931.9028114186859
Linear Regression (Unscaled) Cross validation R2 Mean : 0.7506118061709335
Ridge Regression R2 Score(Unscaled) : 0.6774339114568133
MSE_ridge(Unscaled): 932.9123204846987
R2 Score(Scaled): 0.6777829618270872
MSE(Scaled): 93

After re-modelling using datasets where **low-variance columns were removed** The  results show that removing low variance columns **has almost no effect on model performance across most datasets**, with R² and MSE values remaining largely unchanged. This shows that such features were either redundant but may carry minimal predictive value except  in **FD003** where  removing these columns reduced performance,**showing that even low variance features can carry relevant signals**. Comparing linear and ridge regression further confirms that **regularization does not  improve results**. 
**FD002 shows a marginal gain when scaled ridge regression is applied (R² increasing from 0.678 to 0.679 and MSE dropping from ~932 to ~929).** 
In **FD004 ridge regression actually decreases predictive accuracy**, confirming its limited utility in this context.

linear regression remains the most stable baseline model across the datasets. Performance differences between cross-validation and test sets also show a persistent generalization gap, highlighting that test sets are harder to predict . 

#### **CONCLUSIONS**
The findings confirm that linear models, even with regularization and feature selection, are insufficient for capturing the complex and nonlinear degradation dynamics in engine data. Moving forward, non-linear methods such as random forests, gradient boosting, or neural networks are better candidates for improving predictive accuracy. 



In [ ]:
 # Modelling using XG boost
for (name1, df1) , (name2, df2)in zip(train_dfs.items(), test_dfs.items()):
    X_train = df1.iloc[ :,1:-1 ]
    y_train = df1.iloc[ :,-1 ]

    X_test = df2.iloc[ :,1:-1 ]
    y_test = df2.iloc[ :,-1 ]


    model = XGBRegressor(
        n_estimators=100,      # Number of boosting rounds (trees)
        learning_rate=0.1,     # Shrinkage step to prevent overfitting
        max_depth=5,           # Maximum depth of each tree
        subsample=0.8,         # Fraction of data to sample for each tree
        colsample_bytree=0.8,  # Fraction of features for each tree
        random_state=42
    )
    

    reg = xgb.XGBRegressor()

    grid = GridSearchCV(reg, params_grid, scoring = 'accuracy', cv = 5 )
    grid.fit(X_train , y)
